# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umemon254/Fly-rank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My baseline gives each content item an action score based on signals that are available at the decision moment. Items with a worse recent trend receive more points. Low click-through rate and high average search position can also increase the score because they may indicate weaker search performance.

This is a simple decision-support rule, not a claim that an item will definitely decline. The goal is to rank items so the highest-risk items can be reviewed first.

Reason codes
NEGATIVE_TREND — the item's recent trend signal is negative.
LOW_CTR — the item has a relatively low click-through rate.
HIGH_POSITION — the item has a relatively high average search position, meaning it is appearing lower in search results.
MULTIPLE_SIGNALS — more than one weak signal is present.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
!pip -q install duckdb pyarrow huggingface_hub

In [5]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [6]:
import duckdb
import pandas as pd
import numpy as np
import os

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB connection created successfully")

DuckDB connection created successfully


In [7]:
con.execute(f"""
CREATE SECRET hf (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face access configured successfully")

Hugging Face access configured successfully


In [10]:
columns_df = con.execute("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

columns_df

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [12]:
columns_df = con.execute("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print(columns_df["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [15]:
feature_df = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_avg_position,
    gsc_clicks,
    gsc_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE client_has_gsc IS TRUE
  AND gsc_avg_position IS NOT NULL
  AND gsc_impressions > 0
""").df()

print("Feature rows:", len(feature_df))
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 3611061


,report_date,client_hash_id,content_hash_id,gsc_avg_position,gsc_clicks,gsc_impressions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,3.350000,0,20
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,0.000000,0,1
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4.928000,1,125
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,4.000000,0,7
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.272727,0,11


In [16]:
import pandas as pd
import numpy as np
import os

baseline_df = feature_df.copy()

# Create CTR using the warehouse columns
baseline_df["ctr"] = (
    baseline_df["gsc_clicks"] /
    baseline_df["gsc_impressions"]
)

# Remove invalid rows
baseline_df = baseline_df.dropna(
    subset=["ctr", "gsc_avg_position"]
).copy()

# Create thresholds
ctr_low_threshold = baseline_df["ctr"].quantile(0.25)

position_high_threshold = (
    baseline_df["gsc_avg_position"].quantile(0.75)
)

# Start score at zero
baseline_df["action_score"] = 0

# Identify weak signals
baseline_df["low_ctr"] = (
    baseline_df["ctr"] <= ctr_low_threshold
)

baseline_df["high_position"] = (
    baseline_df["gsc_avg_position"] >= position_high_threshold
)

# Add points
baseline_df.loc[
    baseline_df["low_ctr"],
    "action_score"
] += 1

baseline_df.loc[
    baseline_df["high_position"],
    "action_score"
] += 1


# Create reason codes
def make_reason(row):
    reasons = []

    if row["low_ctr"]:
        reasons.append("LOW_CTR")

    if row["high_position"]:
        reasons.append("HIGH_POSITION")

    if len(reasons) > 1:
        reasons.append("MULTIPLE_SIGNALS")

    return "|".join(reasons) if reasons else "NO_STRONG_SIGNAL"


baseline_df["reason_code"] = baseline_df.apply(
    make_reason,
    axis=1
)

# Rank highest priority first
baseline_df = baseline_df.sort_values(
    ["action_score", "gsc_avg_position"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1


# Final output
baseline_output = baseline_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_clicks",
        "gsc_impressions",
        "ctr",
        "gsc_avg_position",
        "action_score",
        "reason_code"
    ]
]

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows ranked:", len(baseline_output))
print("Low CTR threshold:", ctr_low_threshold)
print("High position threshold:", position_high_threshold)

baseline_output.head(20)

Rows ranked: 3611061
Low CTR threshold: 0.0
High position threshold: 20.2


,rank,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,ctr,gsc_avg_position,action_score,reason_code
0,1,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,2026-03-14,0,1,0.0,498.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
1,2,client_23a62021009f63c4,content_13ef8874a9a1ef5e,2026-03-22,0,1,0.0,497.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
2,3,client_23a62021009f63c4,content_aacb637aa920b8ea,2026-03-30,0,1,0.0,495.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
3,4,client_23a62021009f63c4,content_c7ebdf81f488f0d8,2026-03-23,0,1,0.0,480.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
4,5,client_20259bd6705d81d4,content_8c34799f566ce23c,2026-03-03,0,1,0.0,469.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
5,6,client_23a62021009f63c4,content_6f6a949b6e7eb3ae,2026-03-22,0,1,0.0,465.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
6,7,client_e547b89c05043229,content_aa376cef98a5fae8,2026-03-03,0,1,0.0,447.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
7,8,client_23a62021009f63c4,content_070944208ef3a890,2026-03-22,0,1,0.0,445.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
8,9,client_23a62021009f63c4,content_74e1dddeda79c23c,2026-03-31,0,1,0.0,444.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
9,10,client_20259bd6705d81d4,content_9afdc38dbabc43c6,2026-03-15,0,2,0.0,403.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
top20 = baseline_output.head(20).copy()

top20

,rank,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,ctr,gsc_avg_position,action_score,reason_code
0,1,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,2026-03-14,0,1,0.0,498.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
1,2,client_23a62021009f63c4,content_13ef8874a9a1ef5e,2026-03-22,0,1,0.0,497.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
2,3,client_23a62021009f63c4,content_aacb637aa920b8ea,2026-03-30,0,1,0.0,495.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
3,4,client_23a62021009f63c4,content_c7ebdf81f488f0d8,2026-03-23,0,1,0.0,480.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
4,5,client_20259bd6705d81d4,content_8c34799f566ce23c,2026-03-03,0,1,0.0,469.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
5,6,client_23a62021009f63c4,content_6f6a949b6e7eb3ae,2026-03-22,0,1,0.0,465.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
6,7,client_e547b89c05043229,content_aa376cef98a5fae8,2026-03-03,0,1,0.0,447.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
7,8,client_23a62021009f63c4,content_070944208ef3a890,2026-03-22,0,1,0.0,445.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
8,9,client_23a62021009f63c4,content_74e1dddeda79c23c,2026-03-31,0,1,0.0,444.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS
9,10,client_20259bd6705d81d4,content_9afdc38dbabc43c6,2026-03-15,0,2,0.0,403.0,2,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS


I reviewed the top 20 items produced by the baseline. For each item, the proposed action is to review the content first rather than immediately change it. The confidence is higher when multiple signals agree. A pick could be wrong because the baseline uses a small number of signals and does not know the full reason behind a change in performance.

In [18]:
review_rows = []

for _, row in top20.iterrows():
    reason = row["reason_code"]
    score = row["action_score"]

    if score >= 2:
        confidence = "Medium — more than one weak signal is present."
    elif score == 1:
        confidence = "Low to medium — only one signal triggered the rule."
    else:
        confidence = "Low — the baseline found no strong signal."

    review_rows.append({
        "rank": row["rank"],
        "action": "Review this content first",
        "reason_code": reason,
        "confidence_note": confidence,
        "what_would_make_it_wrong":
            "The weak signal may be temporary, seasonal, caused by measurement differences, or explained by information not included in this baseline."
    })

top20_review = pd.DataFrame(review_rows)

top20_review

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
1,2,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
2,3,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
3,4,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
4,5,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
5,6,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
6,7,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
7,8,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
8,9,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."
9,10,Review this content first,LOW_CTR|HIGH_POSITION|MULTIPLE_SIGNALS,Medium — more than one weak signal is present.,"The weak signal may be temporary, seasonal, ca..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
print("Columns used to calculate the score:")
print([
    "ctr",
    "gsc_avg_position"
])

print("\nChecking for possible leakage columns...")

leakage_words = [
    "label",
    "future",
    "outcome",
    "flag",
    "action_taken"
]

possible_leakage = [
    col for col in baseline_df.columns
    if any(word in col.lower() for word in leakage_words)
]

print("Possible leakage-related columns found:")
print(possible_leakage)

print("\nBaseline features actually used:")
print("ctr")
print("gsc_avg_position")

Columns used to calculate the score:
['ctr', 'gsc_avg_position']

Checking for possible leakage columns...
Possible leakage-related columns found:
[]

Baseline features actually used:
ctr
gsc_avg_position


Some picks may be weak because the baseline only uses a small number of search signals. For example, an item can have a low CTR because of a temporary change, the type of search query, or differences in how users interact with that content. A high average position can also be misleading when the item receives very little search traffic.

I also checked the baseline for leakage. I did not use product flags, future outcome columns, labels, or columns created from future performance. The IDs are only kept for identifying and reviewing the ranked items, not used to calculate the action score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.